<a href="https://colab.research.google.com/github/leminhohoho/context-bert4rec/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install relbench torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.8 MB/s eta 0:00:00


In [2]:
import relbench
from relbench.tasks import get_task
from relbench.datasets import get_dataset

dataset = get_dataset("rel-amazon", download=True)
task = get_task("rel-amazon", "user-item-purchase", download=True)

train_table = task.get_table("train")
test_table = task.get_table("test")
val_table = task.get_table("val")

print(train_table)
print(test_table)
print(val_table)

100%|█████████████████████████████████████| 6.40G/6.40G [00:00<00:00, 6.02TB/s]
Unzipping contents of '/root/.cache/relbench/rel-amazon/db.zip' to '/root/.cache/relbench/rel-amazon/.'
100%|█████████████████████████████████████| 75.5M/75.5M [00:00<00:00, 85.2GB/s]
Unzipping contents of '/root/.cache/relbench/rel-amazon/tasks/user-item-purchase.zip' to '/root/.cache/relbench/rel-amazon/tasks/.'


Table(df=
         timestamp  customer_id                            product_id
0       2014-04-03       393192                        [80225, 12439]
1       2014-04-03       149856  [38972, 29137, 422124, 61124, 80784]
2       2014-04-03       384140               [40744, 129519, 410405]
3       2014-04-03       747159                 [26162, 22215, 26195]
4       2014-04-03       318496                               [12321]
...            ...          ...                                   ...
5112798 2014-04-03      1396478                              [145153]
5112799 2014-04-03       244375                                [2373]
5112800 2014-04-03      1352968                              [319970]
5112801 2014-04-03      1459617                              [245873]
5112802 2014-04-03      1561655                              [409234]

[5112803 rows x 3 columns],
  fkey_col_to_pkey_table={'customer_id': 'customer', 'product_id': 'product'},
  pkey_col=None,
  time_col=timestamp)
Tab

In [3]:
import torch.nn as nn
import torch

In [4]:
dev = True

In [5]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()

        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        batch_size = x.size(0)
        return self.pe.weight.unsqueeze(0).repeat(batch_size, 1, 1)

class BERT4RecEmbedding(nn.Module):
    def __init__(self, embed_size, max_len, dropout=0.1):
        super().__init__()

        self.pe = PositionalEmbedding(max_len, embed_size)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        pos = self.pe(x)
        return self.dropout(x + pos)

if __name__ == "__main__" and dev:
    max_len = 5
    d_model = 4
    batch_size = 2

    model = BERT4RecEmbedding(d_model,max_len)
    x = torch.randn(batch_size, max_len, d_model)

    out = model(x)

    print(x.shape)
    print(x)
    print(out.shape)
    print(out)

    # NOTE: Padding example
    x = torch.randn(batch_size, max_len-2, d_model)
    x = torch.nn.functional.pad(x, (0 , 0, 0, 2))
    padding_mask = (x.abs().sum(dim=-1) == 0)

    print(x.shape)
    print(x)
    print(padding_mask.shape)
    print(padding_mask)


torch.Size([2, 5, 4])
tensor([[[ 0.9725,  0.9213,  1.6845, -0.7117],
         [ 0.1168,  0.0468,  1.6997, -0.5614],
         [ 0.1162,  0.3170, -0.0830, -1.6572],
         [ 0.4684,  1.5628,  0.3634,  1.5144],
         [-0.7612,  0.6337,  1.9283, -0.4439]],

        [[-0.6761,  0.3889, -0.7118,  0.4733],
         [ 0.7865,  0.3445,  1.4314, -1.3451],
         [-1.7541,  0.6082,  0.7528,  0.1242],
         [-0.5105,  1.3440, -0.5402, -2.1111],
         [-0.2825, -0.1966, -0.0595, -0.9392]]])
torch.Size([2, 5, 4])
tensor([[[ 0.0000,  1.2188,  1.3363, -1.6720],
         [-0.0000, -0.5140,  1.9729, -0.0000],
         [ 1.8282,  0.0000,  0.2133, -0.0000],
         [ 0.6545,  2.3849,  1.9786,  1.5078],
         [-0.7932,  0.7310,  1.2485, -0.4148]],

        [[-1.5674,  0.6274, -0.0000, -0.3553],
         [-0.8428, -0.0000,  1.6749, -1.5862],
         [-0.2499,  0.6159,  1.1419,  1.4045],
         [-0.4332,  2.1417,  0.9746, -2.5205],
         [-0.0000, -0.1915, -0.9602, -0.0000]]], grad_fn=

In [9]:
class BERT(nn.Module):
    def __init__(self, max_len, n_layers, n_heads, hidden, dropout=0.1):
        super().__init__()
        self.embedding = BERT4RecEmbedding(
            embed_size=hidden,
            max_len=max_len,
            dropout=dropout,
        )

        self.mask_token = nn.Parameter(torch.randn(hidden))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=n_heads,
            batch_first=True,
            dropout=dropout,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=n_layers,
        )

        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, padding_mask=None, masked_positions=None):
        if padding_mask is None:
            padding_mask = (x.abs().sum(dim=-1) == 0)

        x = self.embedding(x)

        if masked_positions is not None:
            x = torch.where(
                masked_positions.unsqueeze(-1),
                self.mask_token.view(1, 1, -1),
                x,
            )

        x = self.encoder(x, src_key_padding_mask=padding_mask)
        x = self.dropout(x)

        return x

if __name__ == "__main__" and dev:
    batch_size = 2
    max_len = 4
    hidden = 32
    n_layers = 3
    n_heads = 4

    model = BERT(max_len=max_len, hidden=hidden, n_layers=n_layers, n_heads=n_heads)

    x = torch.randn(batch_size, max_len, hidden)

    out = model(x)

    print(out.shape)
    print(out)

torch.Size([2, 4, 32])
tensor([[[-2.0732,  0.2100,  0.1621,  1.2629,  0.0742, -1.1898, -1.1427,
           0.7181,  0.0000, -0.7327,  0.2021,  0.4134, -2.1365, -1.0789,
          -0.4325,  0.7241,  0.0000, -2.6345, -0.6530,  1.2156, -0.8531,
           0.8649,  1.4534,  1.9829, -0.4030, -0.5818, -0.2056,  0.5953,
           0.8940, -0.1383,  0.0000,  1.9325],
         [ 1.3745, -0.3223, -1.0846,  0.0905, -1.4864,  0.0815,  0.8295,
           0.5048, -1.9231, -1.0279, -0.4617,  1.5987,  1.4595, -1.6040,
          -1.6533, -0.7296,  0.0521,  1.1358,  0.1179,  0.1802,  1.4575,
          -2.0822, -0.9174, -0.2677,  0.4699,  0.5649, -0.5560,  2.4541,
           0.0000,  0.8816,  0.9611, -0.4889],
         [ 0.5594,  1.4244, -0.8723,  2.5451, -2.2627,  0.5793,  0.8540,
          -0.0000,  1.0184, -0.2555,  0.0299,  0.5057, -1.1021, -0.9717,
           0.4195, -0.1761,  0.0000, -0.1937,  1.3977, -0.9334, -2.1870,
          -1.7858,  1.7929, -0.1987, -0.1275, -0.3227,  0.1004,  0.7163,
       

In [7]:
import torch.nn.functional as F

class UserEncoder(nn.Module):
    def __init__(self, max_len, n_layers, n_heads, hidden, dropout=0.1):
        super().__init__()

        self.encoder = BERT(
            max_len=max_len,
            n_layers=n_layers,
            n_heads=n_heads,
            hidden=hidden,
            dropout=dropout,
        )

        self.proj = nn.Linear(hidden, hidden)

    def forward(self, x):
        x = self.encoder(x)
        x = self.proj(x[:, -1, :])

        return x

class CandidateGenerator(nn.Module):
    def __init__(self, max_len, n_layers, n_heads, hidden, dropout=0.1):
        super().__init__()

        self.user_encoder = UserEncoder(
            max_len=max_len,
            n_layers=n_layers,
            n_heads=n_heads,
            hidden=hidden,
            dropout=dropout,
        )

    def forward(self, user_seq, targets):
        print(user_seq.shape)
        user_embedding = self.user_encoder(user_seq)
        user_embedding = user_embedding.unsqueeze(1)
        user_embedding = F.normalize(user_embedding, dim=-1)
        targets = F.normalize(targets, dim=-1)
        targets = targets.transpose(1,2)

        print(user_embedding.shape)
        print(targets.shape)

        logits = torch.bmm(user_embedding, targets)

        return logits

if __name__ == "__main__" and dev:
    batch_size = 2
    max_len = 4
    hidden = 16
    n_layers = 3
    n_heads = 4
    items_length = 100

    model = CandidateGenerator(max_len=max_len, hidden=hidden, n_layers=n_layers, n_heads=n_heads)

    x = torch.randn(batch_size, max_len, hidden)
    targets = torch.randn(batch_size, items_length, hidden)

    out = model(x, targets)

    print(out.shape)
    print(out)

torch.Size([2, 4, 16])
torch.Size([2, 1, 16])
torch.Size([2, 16, 100])
torch.Size([2, 1, 100])
tensor([[[-2.8081e-01,  4.0510e-01, -1.1166e-01,  3.3629e-02, -5.9342e-02,
           3.3153e-01,  2.9311e-01, -2.9220e-02, -8.7419e-02, -7.3385e-02,
          -3.5483e-01,  3.4901e-01,  8.7404e-02,  2.9114e-01, -1.8915e-01,
           5.3595e-02, -3.2330e-02,  3.9078e-02,  2.2207e-01,  2.8803e-01,
          -3.2871e-01, -1.9345e-01,  1.0308e-01, -8.6288e-04, -1.1357e-01,
           2.6547e-01,  1.4580e-01, -6.4071e-02,  1.9432e-01,  8.7227e-02,
           1.4581e-01,  4.2726e-01, -1.7742e-02,  2.2025e-01, -2.2181e-01,
           1.1160e-01,  7.1093e-03, -2.6070e-01, -2.1594e-01, -3.4886e-01,
           2.6390e-01,  1.6513e-01, -1.6432e-01,  6.3550e-01,  1.0955e-01,
          -3.1449e-01,  3.5495e-01,  2.8601e-04, -1.0712e-01, -2.3887e-01,
          -4.8015e-01,  1.9357e-01, -4.5280e-01,  2.1610e-01,  6.3929e-02,
           3.8696e-02, -1.8228e-01,  8.1538e-02, -4.3873e-02,  2.7817e-01,
     

In [10]:
db = dataset.get_db()

Loading Database object from /root/.cache/relbench/rel-amazon/db...
Done in 63.79 seconds.


In [ ]:
customers_df = db.table_dict["customer"].df
products_df = db.table_dict["product"].df
reviews_df = db.table_dict["review"].df

print(customers_df.head())
print(customers_df.shape)
print("------------------------")
print(products_df.head().to_string())
print(products_df.shape)
print("------------------------")
print(reviews_df.head().to_string())
print(reviews_df.shape)

print(train_table.df)

NameError: name 'db' is not defined

In [ ]:
def get_user_seq(row):
    user_past_reviews = reviews_df[(reviews_df["review_time"] < row["timestamp"]) & (reviews_df["customer_id"] == row["customer_id"])]

    return products_df.set_index("product_id").join(user_past_reviews, on="product_id")

print(get_user_seq(train_table.df.iloc[0]).to_string())
print(train_table.df.iloc[0]["product_id"])